# Pruebas con los algoritmos · borrador

La cocina de los modelos. Aquí se prueba rápido y en sucio lo que luego se lleva en
limpio a `src/`: cuánto tarda cada modelo, cuánta nota regalaban las columnas de fuga y
si la red de Keras se reinicia de verdad en cada `fit`.

**Este notebook no se defiende y no tiene que estar bonito**, pero sus números son los
que cita el README, y aquí se pueden volver a sacar.

Para no repetirse, usa `data_loader.preparar()` en lugar de volver a cargar el CSV a mano.

In [1]:
import sys, pathlib

# Subir hasta la raiz del proyecto (la carpeta que contiene src/), la abras desde donde la abras.
raiz = next(d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (d / "src").is_dir())
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

import time

import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline

from src import config, data_loader, model_trainer

pd.set_option("display.max_columns", 40)

d = data_loader.preparar()
X_train, y_train = d["X_train"], d["y_train"]
print(X_train.shape)

      [limpieza] fugas: -4 columnas (reservation_status, reservation_status_date, required_car_parking_spaces, assigned_room_type)
      [limpieza] duplicados exactos: -33.413 filas
      [limpieza] adr negativo: -1 filas
      [limpieza] 0 huéspedes: -165 filas
      [limpieza] total: 119.390 -> 85.811 filas
      train 68.648 / test 17.163 filas | cancelan: 27.64% en train, 27.64% en test
(68648, 27)


## 1. Muestra pequeña para iterar rápido

68.648 filas x 5 folds x 6 modelos es una espera larga para probar una idea. Se prueba
con una muestra estratificada; el resto del train sirve para validar (el test no se toca).

In [2]:
X_mini, X_resto, y_mini, y_resto = train_test_split(
    X_train, y_train, train_size=10_000, stratify=y_train, random_state=config.SEMILLA,
)
print(X_mini.shape, round(y_mini.mean(), 4))

(10000, 27) 0.2764


## 2. Cuánto tarda cada modelo

Dato práctico: decide qué entra en el modo `--demo` y qué se deja fuera de la búsqueda
de hiperparámetros.

In [3]:
tiempos = {}
for nombre in config.MODELOS_ACTIVOS:
    pipeline = Pipeline([("prep", clone(d["preprocesador"])),
                         ("modelo", model_trainer.REGISTRO[nombre]())])
    inicio = time.perf_counter()
    pipeline.fit(X_mini, y_mini)
    tiempos[nombre] = time.perf_counter() - inicio

pd.Series(tiempos).round(1).sort_values().to_frame("segundos (un fit, 10.000 filas)")

,"segundos (un fit, 10.000 filas)"
baseline,0.0
logistica,0.1
arbol,0.1
bosque,0.4
boosting,0.9
red_keras,21.1


La red es con diferencia la más lenta, y además va en un solo proceso. Por eso se
queda fuera de la búsqueda de hiperparámetros y en `--demo` entrena menos épocas.

## 3. Las fugas posteriores: cuánta nota regalan

`required_car_parking_spaces` y `assigned_room_type` se rellenan cuando el cliente llega
(ver `eda_final.ipynb`, sección 2). Para medir cuánto inflan la nota se comparan las
MISMAS filas del train con y sin ellas, con un gradient boosting de scikit-learn por
defecto (salvo la semilla) y los mismos folds que el pipeline. Es la cifra del apartado 2
del README.

In [4]:
crudo = data_loader.cargar_crudo()

# Las mismas filas que deja limpiar(), pero conservando las columnas de fuga.
filas = crudo.loc[~crudo.drop(columns=config.FUGAS).duplicated()]
huespedes = filas[["adults", "children", "babies"]].fillna(0).sum(axis=1)
filas = filas[(filas["adr"] >= 0) & (huespedes > 0)].reset_index(drop=True)
assert filas.drop(columns=config.FUGAS).equals(data_loader.limpiar(crudo))

X_con_fugas = filas.loc[X_train.index, [*X_train.columns, *config.FUGAS_POSTERIORES]]
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.SEMILLA)

def f1_y_auc(X):
    pipeline = Pipeline([
        ("prep", data_loader.construir_preprocesador(X)),
        ("modelo", HistGradientBoostingClassifier(random_state=config.SEMILLA)),
    ])
    r = cross_validate(pipeline, X, y_train, cv=folds, scoring=["f1", "roc_auc"], n_jobs=-1)
    return round(r["test_f1"].mean(), 3), round(r["test_roc_auc"].mean(), 3)

print("sin las dos (F1, AUC):", f1_y_auc(X_train))
print("con las dos (F1, AUC):", f1_y_auc(X_con_fugas))

      [limpieza] fugas: -4 columnas (reservation_status, reservation_status_date, required_car_parking_spaces, assigned_room_type)
      [limpieza] duplicados exactos: -33.413 filas
      [limpieza] adr negativo: -1 filas
      [limpieza] 0 huéspedes: -165 filas
      [limpieza] total: 119.390 -> 85.811 filas


sin las dos (F1, AUC): (0.678, 0.897)


con las dos (F1, AUC): (0.71, 0.914)


Con ellas el F1 sube unas tres centésimas sin que el modelo sepa nada nuevo: es nota
regalada, porque cuando toca predecir nadie tiene aún plaza anotada ni habitación
asignada. Se quitan (`config.FUGAS_POSTERIORES`).

## 4. ¿Y country?

`country` es la variable que más pesa en el ganador (`outputs/feature_importance.png`).
Tiene algo raro: en los No-Show se parece más a las canceladas que a las que llegaron,
el mismo patrón que delató al parking.

In [5]:
print("sin country (F1, AUC):", f1_y_auc(X_train.drop(columns="country")))

portugal = filas["country"].eq("PRT")
display(portugal.groupby(filas["reservation_status"]).mean().mul(100).round(1)
        .to_frame("% de reservas de Portugal"))

sin country (F1, AUC): (0.59, 0.839)


,% de reservas de Portugal
reservation_status,
Canceled,39.6
Check-Out,27.6
No-Show,59.4


Quitarla cuesta mucho F1 y no hay una prueba tan clara como la del parking (ninguna
reserva con plaza estaba cancelada; aquí solo cambia la proporción). **Se queda**, y la
duda va declarada en el apartado 10 del README: si el país se corrige en el check-in,
parte de su peso sería información que no existe al reservar.

## 5. La red de Keras: la trampa del proyecto

La red se construye en `fit`, **nunca** en `__init__`. Si se construye en `__init__`,
`clone()` reparte el mismo objeto de Keras a los cinco folds y, como el `fit` de Keras no
reinicia los pesos, el fold 2 arranca habiendo visto ya sus datos de validación. El F1
sale inflado y no salta ningún error. Comprobación: dos `fit` seguidos tienen que dar
exactamente lo mismo.

In [6]:
red = Pipeline([("prep", clone(d["preprocesador"])),
                ("modelo", model_trainer.RedKeras(epocas=5))])
f1s = [f1_score(y_resto, red.fit(X_mini, y_mini).predict(X_resto)) for _ in range(2)]
print("F1 de dos fit seguidos:", f1s, "| iguales:", f1s[0] == f1s[1])

# Lo que pasaría con la trampa: seguir entrenando la MISMA red otras 5 épocas.
keras_red = red.named_steps["modelo"].model_
keras_red.fit(red[:-1].transform(X_mini).astype("float32"), y_mini.to_numpy(dtype="float32"),
              epochs=5, verbose=0)
print("F1 si no se reconstruye:", f1_score(y_resto, red.predict(X_resto)))

F1 de dos fit seguidos: [0.5394872578611546, 0.5394872578611546] | iguales: True


F1 si no se reconstruye: 0.6226454438034337


## Lo que no se prueba aquí

- La **búsqueda de hiperparámetros** ya no es una prueba: vive en el pipeline
  (`config.BUSQUEDA = "grid"` y `espacio_busqueda()` de cada modelo).
- El **balanceo de clases** (`class_weight`, SMOTE) se dejó fuera a propósito: la métrica
  es F1 con umbral 0,5 y se compara a todos los modelos en igualdad de condiciones.